## Accelerate Inference: Neural Network Pruning

In [146]:
!pip install torchsummary

In [147]:
import os
import numpy as np
import cv2
import pickle
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchsummary import summary

In [148]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [149]:
# untar
!ls
!tar -xvzf dataset.tar.gz
# load train
train_images = pickle.load(open('train_images.pkl', 'rb'))
train_labels = pickle.load(open('train_labels.pkl', 'rb'))
# load val
val_images = pickle.load(open('val_images.pkl', 'rb'))
val_labels = pickle.load(open('val_labels.pkl', 'rb'))

'ls' is not recognized as an internal or external command,
operable program or batch file.
x train_images.pkl
x train_labels.pkl
x val_images.pkl
x val_labels.pkl


In [150]:
train_images = torch.tensor(train_images, dtype=torch.float32)
val_images = torch.tensor(val_images, dtype=torch.float32)

train_images = train_images.permute(0, 3, 1, 2)
val_images = val_images.permute(0, 3, 1, 2)

In [151]:
train_dataset = TensorDataset(train_images,
                              torch.tensor(train_labels.squeeze(), dtype=torch.long))
val_dataset = TensorDataset(val_images,
                            torch.tensor(val_labels.squeeze(), dtype=torch.long))

In [152]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [153]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()

        self.model = nn.Sequential(
            # First block: Conv -> ReLU -> Conv -> ReLU -> MaxPool -> Dropout
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=True),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=0, bias=True),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Second block: Conv -> ReLU -> Conv -> ReLU -> MaxPool -> Dropout
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=True),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=0, bias=True),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Flatten layer
            nn.Flatten(),

            # Fully connected block: Dense -> ReLU -> Dropout -> Dense -> Softmax
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 5),
        )

    def forward(self, x):
        return self.model(x)

In [154]:
model = ConvNet()

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-6)

In [155]:
model = model.to(device)
summary(model, input_size=(3, 25, 25))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 25, 25]             896
              ReLU-2           [-1, 32, 25, 25]               0
            Conv2d-3           [-1, 32, 23, 23]           9,248
              ReLU-4           [-1, 32, 23, 23]               0
         MaxPool2d-5           [-1, 32, 11, 11]               0
           Dropout-6           [-1, 32, 11, 11]               0
            Conv2d-7           [-1, 64, 11, 11]          18,496
              ReLU-8           [-1, 64, 11, 11]               0
            Conv2d-9             [-1, 64, 9, 9]          36,928
             ReLU-10             [-1, 64, 9, 9]               0
        MaxPool2d-11             [-1, 64, 4, 4]               0
          Dropout-12             [-1, 64, 4, 4]               0
          Flatten-13                 [-1, 1024]               0
           Linear-14                  [

In [156]:
def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()  # Set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    # Progress bar for the training loop
    train_loader_tqdm = tqdm(train_loader, desc="Training", leave=False)

    for inputs, labels in train_loader_tqdm:
        optimizer.zero_grad()  # Zero the parameter gradients
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Track loss and accuracy
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        # Update tqdm description with current loss and accuracy
        train_loader_tqdm.set_postfix(loss=running_loss / total, accuracy=100 * correct / total)

    train_accuracy = 100 * correct / total
    train_loss = running_loss / len(train_loader)
    return train_loss, train_accuracy

In [157]:
def validate(model, val_loader, criterion, device):
    model.eval()  # Set model to evaluation mode
    val_loss = 0.0
    correct = 0
    total = 0

    # Progress bar for the validation loop
    val_loader_tqdm = tqdm(val_loader, desc="Validation", leave=False)

    with torch.no_grad():  # Disable gradient calculations for validation
        for inputs, labels in val_loader_tqdm:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Track loss and accuracy
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            # Update tqdm description with current validation loss and accuracy
            val_loader_tqdm.set_postfix(loss=val_loss / total, accuracy=100 * correct / total)

    val_accuracy = 100 * correct / total
    val_loss = val_loss / len(val_loader)
    return val_loss, val_accuracy

In [158]:
# Main training loop
num_epochs = 50
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")

    # Training
    train_loss, train_accuracy = train_one_epoch(model, train_loader, optimizer, criterion, device)

    # Validation
    val_loss, val_accuracy = validate(model, val_loader, criterion, device)

    # Print epoch results
    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, '
          f'Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')

Epoch 1/50


Epoch [1/50], Train Loss: 1.5155, Train Acc: 30.64%, Val Loss: 1.4207, Val Acc: 36.63%
Epoch 2/50


Epoch [2/50], Train Loss: 1.3888, Train Acc: 39.72%, Val Loss: 1.3454, Val Acc: 41.90%
Epoch 3/50


Epoch [3/50], Train Loss: 1.3389, Train Acc: 42.80%, Val Loss: 1.2797, Val Acc: 45.94%
Epoch 4/50


Epoch [4/50], Train Loss: 1.3023, Train Acc: 44.44%, Val Loss: 1.2575, Val Acc: 47.37%
Epoch 5/50


...

Epoch [46/50], Train Loss: 0.7634, Train Acc: 70.69%, Val Loss: 0.8156, Val Acc: 67.72%
Epoch 47/50


Epoch [47/50], Train Loss: 0.7540, Train Acc: 71.24%, Val Loss: 0.8047, Val Acc: 68.40%
Epoch 48/50


Epoch [48/50], Train Loss: 0.7528, Train Acc: 70.94%, Val Loss: 0.8108, Val Acc: 67.76%
Epoch 49/50


Epoch [49/50], Train Loss: 0.7361, Train Acc: 71.94%, Val Loss: 0.8122, Val Acc: 67.96%
Epoch 50/50


Epoch [50/50], Train Loss: 0.7292, Train Acc: 72.20%, Val Loss: 0.7911, Val Acc: 68.44%


In [17]:
torch.save(model.state_dict(), 'my_model_weights_1.pt', _use_new_zipfile_serialization=False)

In [159]:
torch.save(model.state_dict(), 'original_model.pt')
print("Saved trained model for future pruning.")

Saved trained model for future pruning.


### Zeroing weights

Example on how to set weights to zero. You should determine which weights to set to zero using the approaches you defined in the form.

In [18]:
# Set weights that are less than 0.5 to zero
with torch.no_grad():  # disable gradient tracking for efficiency
    for name, param in model.named_parameters():
        if "weight" in name:  # only apply to weights, skip biases
            param[param < 0.9] = 0

In [19]:
val_loss, val_accuracy = validate(model, val_loader, criterion, device)

In [20]:
val_accuracy
#score = (accuracy + num zero weights / total parameters) / 2

19.247524752475247

In [ ]:
torch.save(model.state_dict(), 'my_model_weights_2.pt', _use_new_zipfile_serialization=False)

## **Activation-based pruning**

In [695]:
# reload full model
model = ConvNet().to(device)
model.load_state_dict(torch.load('original_model.pt'))
print("Loaded original model weights.")


Loaded original model weights.


C:\Users\nickc\AppData\Local\Temp\ipykernel_46828\3605758991.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('original_model.pt'))


In [708]:
activation_storage = {}

def get_activation_hook(name):
    def hook(model, input, output):
        #mean_activations = output.detach().mean(dim=(2, 3))  # [batch, channels]
        max_activations = output.detach().amax(dim=(2, 3))  # shape: [batch, channels]

        if name not in activation_storage:
            activation_storage[name] = []
        #activation_storage[name].append(mean_activations.cpu())
        activation_storage[name].append(max_activations.cpu())
    return hook

In [697]:
hooks = []
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d):
        hooks.append(module.register_forward_hook(get_activation_hook(name)))


In [698]:
model.eval()
with torch.no_grad():
    for i, (inputs, _) in enumerate(train_loader):
        if i >= 200:  # subset of training data
            break
        inputs = inputs.to(device)
        _ = model(inputs)

In [699]:
avg_activations = {}
for layer_name, activations in activation_storage.items():
    all_acts = torch.cat(activations, dim=0)  # [num_samples, channels]
    avg_per_filter = all_acts.mean(dim=0)     # [channels]
    avg_activations[layer_name] = avg_per_filter

# recommended to clean up hooks
for h in hooks:
    h.remove()

In [700]:
k_percent = 0.10 # prune bottom X%
pruned_indices = {}

for name, module in model.named_modules():
    if isinstance(module, torch.nn.Conv2d) and name in avg_activations:
        avg = avg_activations[name]
        num_filters = avg.shape[0]
        num_prune = int(k_percent * num_filters)
        prune_idx = torch.argsort(avg)[:num_prune]
        pruned_indices[name] = prune_idx

        with torch.no_grad():
            module.weight[prune_idx] = 0
            if module.bias is not None:
                module.bias[prune_idx] = 0

print("Pruning complete")

Pruning complete


In [701]:
weight_masks = {}

for name, module in model.named_modules():
    if isinstance(module, torch.nn.Conv2d):
        mask = torch.ones_like(module.weight)
        if name in pruned_indices:
            idx = pruned_indices[name]
            mask[idx] = 0
        weight_masks[name] = mask

In [702]:
def train_one_epoch_with_mask(model, train_loader, optimizer, criterion, device, weight_masks):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in tqdm(train_loader, desc="Fine-Tuning", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()

        with torch.no_grad():
            for name, module in model.named_modules():
                if isinstance(module, nn.Conv2d) and name in weight_masks:
                    module.weight.grad *= weight_masks[name]
                    if module.bias is not None and module.bias.grad is not None:
                        module.bias.grad *= weight_masks[name][:, 0, 0, 0]

        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total
    train_loss = running_loss / len(train_loader)
    return train_loss, train_acc

In [703]:
val_loss, val_accuracy = validate(model, val_loader, criterion, device)
print(f"Post-Pruning Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")


Post-Pruning Val Loss: 1.3627, Val Accuracy: 48.55%


In [707]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-6)

# fine-tuning epochs
for epoch in range(8):
    train_loss, train_accuracy = train_one_epoch_with_mask(model, train_loader, optimizer, criterion, device, weight_masks)
    val_loss, val_accuracy = validate(model, val_loader, criterion, device)
    print(f'[Fine-Tuning Epoch {epoch+1}] Val Accuracy: {val_accuracy:.2f}%')



[Fine-Tuning Epoch 1] Val Accuracy: 68.08%


[Fine-Tuning Epoch 2] Val Accuracy: 68.08%


[Fine-Tuning Epoch 3] Val Accuracy: 68.16%


[Fine-Tuning Epoch 4] Val Accuracy: 68.32%


[Fine-Tuning Epoch 5] Val Accuracy: 68.16%


[Fine-Tuning Epoch 6] Val Accuracy: 68.28%


[Fine-Tuning Epoch 7] Val Accuracy: 68.00%


[Fine-Tuning Epoch 8] Val Accuracy: 68.71%


In [705]:
val_loss, val_accuracy = validate(model, val_loader, criterion, device)
print(f"Post-Pruning AND post-fine tuning Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")

Post-Pruning AND post-fine tuning Val Loss: 0.8034, Val Accuracy: 68.20%


In [706]:
def count_zeroed_and_total_weights(model):
    total_params = 0
    zero_params = 0

    for name, param in model.named_parameters():
        if "weight" in name:
            total_params += param.numel()
            zero_params += (param == 0).sum().item()

    print(f"Total weights     : {total_params:,}")
    print(f"Zeroed weights    : {zero_params:,}")
    print(f"Percentage pruned : {100 * zero_params / total_params:.2f}%")
    return total_params, zero_params

total_params, zero_params = count_zeroed_and_total_weights(model)


print(f"k percent: {k_percent}")
print(f"Final Validation Accuracy: {val_accuracy}")
#score = (accuracy + num zero weights / total parameters) / 2

score = (val_accuracy/100 + zero_params / total_params ) / 2
print("Score must be .36 or higher")
print(f"Final score for assignment: {score}")

Total weights     : 592,224
Zeroed weights    : 6,129
Percentage pruned : 1.03%
k percent: 0.1
Final Validation Accuracy: 68.1980198019802
Score must be .36 or higher
Final score for assignment: 0.3461646613376688


with masked weights and average activations:

subset 200, k = 0.90: Val acc = 19.25, score = 0.14526, % pruned = 9.81%
subset 200, k = 0.80: Val acc = 29.35, score = 0.19058, % pruned = 8.77%
subset 200, k = 0.70: Val acc = 37.23, score = 0.22408, % pruned = 7.59%
subset 200, k = 0.60: Val acc = 43.64, score = 0.25099, % pruned = 6.55%
subset 200, k = 0.50: Val acc = 53.98, score = 0.29750, % pruned = 5.52%
subset 200, k = 0.40: Val acc = 60.12, score = 0.32202, % pruned = 4.29%
subset 200, k = 0.30: Val acc = 63.12, score = 0.33190, % pruned = 3.25%
subset 200, k = 0.20: Val acc = 65.12, score = 0.33727, % pruned = 2.07%
subset 200, k = 0.10: Val acc = 67.45, score = 0.34240, % pruned = 1.03%
subset 200, k = 0.05: Val acc = 68.95, score = 0.34721, % pruned = 0.49%
subset 200, k = 0.00: Val acc = 69.66, score = 0.34832, % pruned = 0.00%

With masked weights and max activation:

subset 200, k = 0.20: Val acc = 67.36, score = 0.34718, % pruned = 2.07%
subset 200, k = 0.10: Val acc = 68.20, score = 0.34616, % pruned = 1.03%


Without masked weights:

subset 200, k = 0.45: Val acc = 58.38, score = 0.31803
subset 200, k = 0.35: Val acc = 59.19, score = 0.32575
subset 200, k = 0.30: Val acc = 61.90, score = 0.32575
subset 200, k = 0.25: Val acc = 62.81, score = 0.32785
subset 200, k = 0.22: Val acc = 63.37, score = 0.3289
subset 200, k = 0.20: Val acc = 64.20, score = 0.3313
subset 200, k = 0.19: Val acc = 63.88, score = 0.32975
subset 200, k = 0.10: Val acc = 67.72, score = 0.34279
subset 200, k = 0.00: Val acc = 69.54, score = 0.34772


## **Modified Network Slimming**


In [626]:
class SlimmableConvNet(nn.Module):
    def __init__(self, linear_input_dim=1024):
        super(SlimmableConvNet, self).__init__()

        self.model = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=True),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=0, bias=True),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=True),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=0, bias=True),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            nn.Flatten(),
            nn.Linear(linear_input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 5),
        )

    def forward(self, x):
        return self.model(x)


In [632]:
def compute_flatten_dim(model, input_shape=(3, 25, 25)):
    with torch.no_grad():
        dummy_input = torch.randn(1, *input_shape)
        output = model.model[:-5](dummy_input)  # skip FC layers
        return output.view(1, -1).shape[1]

In [ ]:
def get_bn_pruning_masks(model_with_bn, threshold=0.01):
    masks = []
    for module in model_with_bn.modules():
        if isinstance(module, nn.BatchNorm2d):
            gamma = module.weight.data.abs().clone()
            mask = gamma > threshold
            masks.append(mask)
    return masks

In [618]:
def prune_conv2d_weights(conv, mask_out, mask_in=None):
    W = conv.weight.data.clone()
    B = conv.bias.data.clone() if conv.bias is not None else None

    W = W[mask_out]
    if B is not None:
        B = B[mask_out]
    if mask_in is not None:
        W = W[:, mask_in, :, :]
    return W, B

In [657]:
def get_conv_layers_only(model):
    return [m for m in model.modules() if isinstance(m, nn.Conv2d)]

In [658]:
def transfer_pruned_weights(model_with_bn, model_orig, masks):
    conv_bn_layers = get_conv_layers_only(model_with_bn)
    conv_orig_layers = get_conv_layers_only(model_orig)

    mask_in = None
    for i, (conv_bn, conv_orig, mask_out) in enumerate(zip(conv_bn_layers, conv_orig_layers, masks)):
        W, B = prune_conv2d_weights(conv_bn, mask_out, mask_in)
        conv_orig.weight.data.copy_(W)
        if B is not None:
            conv_orig.bias.data.copy_(B)
        mask_in = mask_out

In [ ]:
def train_one_epoch_bn(model, train_loader, optimizer, criterion, device, l1_strength=1e-4):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    train_loader_tqdm = tqdm(train_loader, desc="Training", leave=False)

    for inputs, labels in train_loader_tqdm:
        optimizer.zero_grad()
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # add L1 penalty on batchNorm weights
        l1_loss = 0.0
        for module in model.modules():
            if isinstance(module, nn.BatchNorm2d):
                l1_loss += torch.norm(module.weight, 1)  # gamma values

        # add the L1 loss scaled by lambda
        loss += l1_strength * l1_loss

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        train_loader_tqdm.set_postfix(loss=running_loss / total, accuracy=100 * correct / total)

    train_accuracy = 100 * correct / total
    train_loss = running_loss / len(train_loader)
    return train_loss, train_accuracy


In [613]:
def train_slimmable_model(train_loader, val_loader, device, num_epochs=50):
    # Initialize model
    model = SlimmableConvNet().to(device)

    # Define optimizer and loss
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-6)
    criterion = nn.CrossEntropyLoss()

    # Loop through epochs
    for epoch in range(num_epochs):
        print(f"\nEpoch (bn) {epoch + 1}/{num_epochs}")

        train_loss, train_acc = train_one_epoch_bn(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)

        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc:.2f}%")

    # Save trained model
    torch.save(model.state_dict(), "slim_model_with_bn.pth")
    print("Saved trained SlimmableConvNet for pruning.")

    return model

In [628]:
model_bn = train_slimmable_model(train_loader, val_loader, device, num_epochs=50)



Epoch (bn) 1/50


Training:   0%|          | 0/703 [00:00<?, ?it/s]

Train Loss: 1.4079, Train Acc: 39.60%
Val   Loss: 1.2167, Val   Acc: 48.51%

Epoch (bn) 2/50


Train Loss: 1.2461, Train Acc: 48.88%
Val   Loss: 1.1208, Val   Acc: 53.98%

Epoch (bn) 3/50


Train Loss: 1.1682, Train Acc: 52.93%
Val   Loss: 1.0731, Val   Acc: 55.68%

Epoch (bn) 4/50


...

Train Loss: 0.4992, Train Acc: 81.97%
Val   Loss: 0.6431, Val   Acc: 76.20%

Epoch (bn) 47/50


Train Loss: 0.4813, Train Acc: 82.92%
Val   Loss: 0.6606, Val   Acc: 76.91%

Epoch (bn) 48/50


Train Loss: 0.4721, Train Acc: 83.05%
Val   Loss: 0.6604, Val   Acc: 75.80%

Epoch (bn) 49/50


Train Loss: 0.4704, Train Acc: 82.95%
Val   Loss: 0.6288, Val   Acc: 76.87%

Epoch (bn) 50/50


Train Loss: 0.4663, Train Acc: 83.21%
Val   Loss: 0.6612, Val   Acc: 76.63%
Saved trained SlimmableConvNet for pruning.


In [620]:
def count_zero_weights(model):
    total_weights = 0
    zero_weights = 0

    for param in model.parameters():
        if param.requires_grad:
            total_weights += param.numel()
            zero_weights += torch.sum(param == 0).item()

    percent_pruned = 100.0 * zero_weights / total_weights
    return total_weights, zero_weights, percent_pruned


In [ ]:
def zero_out_pruned_weights(model, masks):
    conv_layers = [m for m in model.model if isinstance(m, nn.Conv2d)]
    mask_in = None

    for conv, mask_out in zip(conv_layers, masks):
        with torch.no_grad():
            # zero out pruned output channels
            out_indices = (~mask_out).nonzero(as_tuple=True)[0]
            conv.weight[out_indices] = 0
            if conv.bias is not None:
                conv.bias[out_indices] = 0

            # zero out pruned input channels if not the first layer
            if mask_in is not None:
                in_indices = (~mask_in).nonzero(as_tuple=True)[0]
                conv.weight[:, in_indices] = 0

        # save current mask as input mask for next layer
        mask_in = mask_out


In [654]:
def get_identity_masks(model_with_bn):
    masks = []
    for module in model_with_bn.model:
        if isinstance(module, nn.BatchNorm2d):
            gamma = module.weight.data
            mask = torch.ones_like(gamma).bool()  # keep everything
            masks.append(mask)
    return masks

In [682]:
model_with_bn = SlimmableConvNet()
model_with_bn.load_state_dict(torch.load("slim_model_with_bn.pth"))
model_with_bn.eval()

# recompute linear input dim
linear_input_dim = compute_flatten_dim(model_with_bn, input_shape=(3, 25, 25))

# original model
model_orig = ConvNet()

# Get all-True masks (simulate no pruning)
masks = get_identity_masks(model_with_bn)

# Transfer weights (no channels dropped)
transfer_pruned_weights(model_with_bn, model_orig, masks)

# Move to device
model_orig.to(device)

# Validate
val_loss, val_accuracy = validate(model_orig, val_loader, criterion, device)
print(f"No-pruning weight transfer accuracy: {val_accuracy:.2f}%, Loss: {val_loss:.4f}")


C:\Users\nickc\AppData\Local\Temp\ipykernel_46828\641045617.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_with_bn.load_state_dict(torch.load("slim_model_with_bn.

No-pruning weight transfer accuracy: 19.25%, Loss: 1.6099


In [681]:
model_with_bn = SlimmableConvNet()
model_with_bn.load_state_dict(torch.load("slim_model_with_bn.pth"))
model_with_bn.eval()

# get slimming masks based on gamma threshold
masks = get_bn_pruning_masks(model_with_bn, threshold=0.8)

def print_pruned_channels(masks):
    for i, mask in enumerate(masks):
        total = mask.numel()
        kept = mask.sum().item()
        print(f"Layer {i+1}: Kept {int(kept)}/{total} channels ({100 * kept / total:.2f}%)")

print_pruned_channels(masks)


# recompute linear input dim
linear_input_dim = compute_flatten_dim(model_with_bn, input_shape=(3, 25, 25))
#print(linear_input_dim)

# original model
model_orig = ConvNet()

# transfer pruned Conv2d weights into original model
transfer_pruned_weights(model_with_bn, model_orig, masks)

zero_out_pruned_weights(model_orig, masks)

model_orig.to(device)

val_loss, val_accuracy = validate(model_orig, val_loader, criterion, device)
print(f"\nPruned Model Validation Accuracy: {val_accuracy:.2f}%, Loss: {val_loss:.4f}")

total, zeros, pruned_percent = count_zero_weights(model_orig)
print(f"Total weights     : {total:,}")
print(f"Zeroed weights    : {int(zeros):,}")
print(f"Percentage pruned : {pruned_percent:.2f}%")

# # 6. Save the final pruned model (optional)
# torch.save(model_orig.state_dict(), "original_model_pruned_weights.pt")
# print("Pruned weights transferred and saved.")


C:\Users\nickc\AppData\Local\Temp\ipykernel_46828\2390683419.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_with_bn.load_state_dict(torch.load("slim_model_with_bn

Layer 1: Kept 32/32 channels (100.00%)
Layer 2: Kept 32/32 channels (100.00%)
Layer 3: Kept 64/64 channels (100.00%)
Layer 4: Kept 64/64 channels (100.00%)



Pruned Model Validation Accuracy: 20.71%, Loss: 1.6094
Total weights     : 592,933
Zeroed weights    : 0
Percentage pruned : 0.00%
